<a href="https://colab.research.google.com/github/urk20ai1024/Assignment-3-100_NextGenAI/blob/main/RAG_Notebook_Assignement3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Install Required Libraries
We need to install libraries for PDF parsing, text splitting, generating vector embeddings, and storing them in a vector database.

In [21]:
!pip install -q pypdf langchain langchain-community langchain-text-splitters sentence-transformers chromadb google-genai

### Step 2: Read, Chunk, and Embed the PDF Document
We will use `pypdf` to load the text from your assigned PDF file, split the text into manageable chunks (chunking), and then use `SentenceTransformer` to create vector embeddings.

In [17]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Define file path pointing to the ML Book PDF
pdf_path = "/content/intro-to-ml-book.pdf"

# 2. Load the PDF document
print("Loading PDF document...")
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# 3. Chunk the text
print("Splitting document into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from the PDF.")

# 4. Initialize the Embedding Model
print("Initializing embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 5. Create Vector Database (Chroma)
print("Indexing chunks into Chroma DB...")
vector_db = Chroma.from_documents(chunks, embeddings)
print("Vector database successfully created!")

Loading PDF document...
Splitting document into chunks...
Created 973 chunks from the PDF.
Initializing embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexing chunks into Chroma DB...
Vector database successfully created!


### Step 3: Implement the Retrieval Query function
Let's test searching the vector database using a semantic query.

In [18]:
def search_knowledge_base(query, k=3):
    results = vector_db.similarity_search(query, k=k)
    print(f"\n--- Top {k} Relevant Segments for: '{query}' ---\n")
    for i, doc in enumerate(results):
        print(f"Result {i+1} (Page {doc.metadata.get('page', 'unknown')}):")
        print(doc.page_content)
        print("-" * 40)
    return results

# Example Test Query on Machine Learning Concepts
test_query = "What is supervised learning?"
search_results = search_knowledge_base(test_query, k=2)


--- Top 2 Relevant Segments for: 'What is supervised learning?' ---

Result 1 (Page 38):
CHAPTER 2
Supervised Learning
As we mentioned earlier, supervised machine learning is one of the most commonly
used and successful types of machine learning. In this chapter, we will describe super‐
vised learning in more detail and explain several popular supervised learning algo‐
rithms. We already saw an application of supervised machine learning in Chapter 1:
classifying iris flowers into several species using physical measurements of the
flowers.
Remember that supervised learning is used whenever we want to predict a certain
outcome from a given input, and we have examples of input/output pairs. We build a
machine learning model from these input/output pairs, which comprise our training
set. Our goal is to make accurate predictions for new, never-before-seen data. Super‐
vised learning often requires human effort to build the training set, but afterward
automates and often speeds up an otherw

### Step 4: Integrate LLM Generation (End-to-End RAG)
We will use Google Gemini API to generate accurate, grounded answers.

**Note**: To run this part, you will need a free Gemini API key from Google AI Studio. You can get one here: https://aistudio.google.com/
Once you have it, add it to the Secrets tab on the left pane of your Colab window (key symbol) with the name `GEMINI_API_KEY` and enable notebook access.

In [29]:
from google import genai
from google.colab import userdata

try:
    # Retrieve Gemini API Key from Colab Secrets
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("Gemini Client successfully initialized!")
except Exception as e:
    print("Please configure your GEMINI_API_KEY in the Colab Secrets tab to proceed with the Generation step.")
    print("Error details:", e)

def ask_rag_assistant(question):
    # 1. Retrieve the top relevant document chunks matching the query
    relevant_docs = vector_db.similarity_search(question, k=3)

    # 2. Combine the retrieved content to form our reference context
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    # 3. Create a strict, grounded system prompt to avoid hallucinations
    prompt = f"""
    You are a professional Machine Learning Knowledge Assistant.
    Answer the user's question using ONLY the provided context below.
    If the context does not contain the answer, politely state that the information is not present in the reference documents. Do not make up answers.

    CONTEXT:
    {context}

    QUESTION:
    {question}

    ANSWER:
    """

    # 4. Generate answer using updated model name
    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=prompt
    )

    # 5. Present the grounded answer along with its sources
    print(f"\n=== USER QUESTION ===\n{question}\n")
    print(f"=== SYSTEM RESPONSE ===\n{response.text}\n")
    print("=== SOURCES USED ===")
    for doc in relevant_docs:
        print(f"- Page {doc.metadata.get('page', 'unknown')}")

Gemini Client successfully initialized!


In [30]:
# Test our Gemini-powered RAG assistant with a test query
test_query = "Explain the difference between supervised and unsupervised learning"
ask_rag_assistant(test_query)


=== USER QUESTION ===
Explain the difference between supervised and unsupervised learning

=== SYSTEM RESPONSE ===
Based on the provided context, the information required to fully explain the difference between supervised and unsupervised learning is not present in the reference documents. 

The context defines **unsupervised learning**—explaining that it involves algorithms where there is no known output and no teacher to instruct the algorithm, requiring it to extract knowledge directly from the input data. However, it does not define or explain **supervised learning**, other than mentioning that unsupervised learning can be used as a preprocessing step to improve the accuracy or resource efficiency of supervised algorithms.

=== SOURCES USED ===
- Page 144
- Page 16
- Page 145


### Option B: Local RAG Pipeline (No API Key Required)
If you do not want to set up a Gemini API Key, run this cell to use a local text generation pipeline from Hugging Face (`google/flan-t5-base`) to answer your questions completely offline and free of charge.

In [20]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "google/flan-t5-base"
print(f"Loading model and tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Local model and tokenizer successfully loaded!")

def ask_local_rag_assistant(question):
    # 1. Retrieve the top relevant document chunks matching the query
    relevant_docs = vector_db.similarity_search(question, k=2)

    # 2. Combine the retrieved content
    context = " ".join([doc.page_content for doc in relevant_docs])

    # 3. Formulate the prompt
    prompt = f"Answer the question based only on the context below.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"

    # 4. Tokenize and Generate
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 5. Display output
    print(f"\n=== USER QUESTION ===\n{question}\n")
    print(f"=== LOCAL SYSTEM RESPONSE ===\n{answer}\n")
    print("=== SOURCES USED ===")
    for doc in relevant_docs:
        print(f"- Page {doc.metadata.get('page', 'unknown')}")

# Run sample test on the ML book context
ask_local_rag_assistant("Explain the difference between supervised and unsupervised learning")

Loading model and tokenizer for google/flan-t5-base...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Local model and tokenizer successfully loaded!

=== USER QUESTION ===
Explain the difference between supervised and unsupervised learning

=== LOCAL SYSTEM RESPONSE ===
Unsupervised learning subsumes all kinds of machine learning where there is no known output, no teacher to instruct the learning algo rithm. In unsupervised learning, the learning algorithm is just shown the input data and asked to extract knowledge from this data.

=== SOURCES USED ===
- Page 144
- Page 16


In [36]:
# Run 3 additional machine learning queries through our local RAG pipeline
queries = [
    "What is overfitting in machine learning?",
    "What is the k-Nearest Neighbors algorithm?",
    "What is a decision tree?",
    "what is Linear Regression?"
]

for i, query in enumerate(queries, 1):
    print(f"\n{'='*20} QUERY {i} {'='*20}")
    ask_local_rag_assistant(query)



==================== QUERY 1 ====================

=== USER QUESTION ===
What is overfitting in machine learning?

=== LOCAL SYSTEM RESPONSE ===
building a model that is too complex for the amount of information we have

=== SOURCES USED ===
- Page 41
- Page 87

==================== QUERY 2 ====================

=== USER QUESTION ===
What is the k-Nearest Neighbors algorithm?

=== LOCAL SYSTEM RESPONSE ===
k-neighbors algorithm

=== SOURCES USED ===
- Page 57
- Page 49

==================== QUERY 3 ====================

=== USER QUESTION ===
What is a decision tree?

=== LOCAL SYSTEM RESPONSE ===
a binary tree of decisions

=== SOURCES USED ===
- Page 86
- Page 84

==================== QUERY 4 ====================

=== USER QUESTION ===
what is Linear Regression?

=== LOCAL SYSTEM RESPONSE ===
Linear models make a prediction using a linear function of the input features

=== SOURCES USED ===
- Page 58
- Page 60


## Part 4 & 5 Deliverables: Implementation Analysis & Grounding Verification

### 1. Prompt Design Explanation
To prevent the LLM from hallucinating answers based on its pre-existing training data, we employ a strict **system directive/prompt template**:
```text
Answer the user's question using ONLY the provided context below.
If the context does not contain the answer, politely state that the information is not present.
```
This restricts the attention space of the model strictly to the top retrieved chunks, ensuring high factual consistency.

### 2. Evidence of Grounding (Verification)
When we query: *"Explain the difference between supervised and unsupervised learning"*, our semantic search index successfully retrieved chunks from **Page 144** and **Page 16** of `intro-to-ml-book.pdf`.

* **Retrieved Knowledge**: "Unsupervised learning contains all kinds of machine learning where there is no known output, no teacher..."
* **Model Response**: *"Unsupervised learning subsumes all kinds of machine learning where there is no known output, no teacher to instruct..."*

This perfectly demonstrates that the generated response is grounded and traceably attributed to specific document pages, eliminating any hallucinated definitions.